# PACE Phase 11C — AMPGAN v2 source audit and optional benchmark

**Analysis only — do not modify the original PACE manuscript, supplementary data, or reviewer responses.**

This notebook uses the **official AMPGAN v2 `v2.0` tag** (`vail-uvm/amp-gan`, published tag commit starting `2102eb5d`) rather than an unversioned clone. It is intentionally structured as two stages:

**Stage 1:** Clone GitLab's tagged public source and inventory files, checkpoints, generated-sequence candidates and Git LFS pointers. **This does NOT claim AMPGAN v2 candidate output exists.** The source inventory ZIP is produced even when no authenticated generated peptides exist.

**Stage 2 (optional, only after verifying provenance):** Set `RUN_OPTIONAL_BENCHMARK = True` and supply authentic AMPGAN v2 output FASTA/CSV and the already-existing Phase8 data ZIP. The script refits the original-style 300-tree PACE 90D MIC/HC50 predictors for a common **exploratory (non-independent) computational rescoring**.

A PACE-model-based score cannot establish experimental efficacy or fair algorithmic superiority. Do **not** substitute AMPGAN training sequences, AMPGAN v1 output, or AMPGAN v3 model weights for AMPGAN v2 candidates.

Sources: [official repo](https://gitlab.com/vail-uvm/amp-gan), [official tags](https://gitlab.com/vail-uvm/amp-gan/-/tags), [Van Oort et al. paper](https://doi.org/10.1021/acs.jcim.0c01441).

In [ ]:
# Cell 1: Install light dependencies. Colab already includes NumPy/scikit-learn.
%pip install -q biopython
import os, subprocess
from pathlib import Path
WORK=Path('/content')
REPO=WORK/'ampgan_v2_official'
OUT=WORK/'PACE_Phase11C_Output'
OUT.mkdir(exist_ok=True)
print('Environment ready. Original PACE files are not modified.')

In [ ]:
%%writefile /content/pace_phase11c_runner.py
"""PACE Phase 11C, source-authenticity audit + optional sequence-only computational benchmark.
Research analysis only. Never treat training data or generated examples as AMPGAN v2 output
without an authoritatively documented generator run or provenance. Zero edits to user inputs.
"""
from __future__ import annotations
import argparse, collections, csv, hashlib, importlib.util, io, json, os, re, statistics, sys, time, zipfile
from itertools import combinations
from pathlib import Path

AMINO = set('ACDEFGHIKLMNPQRSTVWY')
ENDPOINTS = ['E. faecium', 'S. aureus', 'K. pneumoniae', 'A. baumannii', 'P. aeruginosa', 'E. cloacae']
COUNTS = dict(zip(ENDPOINTS + ['HC50'], [780,4128,743,306,2517,304,1271]))
SOURCE_URI = 'https://gitlab.com/vail-uvm/amp-gan'
RELEASE_TAG = 'v2.0'
RELEASE_SHA_PREFIX = '2102eb5d'


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(2**20), b''): h.update(block)
    return h.hexdigest()


def save_csv(path, rows, fieldnames=None):
    path = Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    names = fieldnames or (list(rows[0]) if rows else ['description'])
    with path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=names, extrasaction='ignore')
        w.writeheader();w.writerows(rows)


def sequences_from_file(path):
    path=Path(path)
    raw=path.read_text(encoding='utf-8-sig',errors='replace')
    typ=path.suffix.lower()
    if typ in ('.csv','.tsv'):
        delimiter='\t' if typ=='.tsv' else ','
        reader=csv.DictReader(io.StringIO(raw),delimiter=delimiter)
        if not reader.fieldnames:raise ValueError('No CSV/TSV header')
        key=next((k for k in reader.fieldnames if k and re.sub(r'[^a-z]','',k.lower()) in ('sequence','peptide','seq','aasequence','peptidesequence')),None)
        if key is None:raise ValueError(f'No standard peptide sequence column in {reader.fieldnames}')
        seqs=[(str(r.get(key,'') or '').strip().upper().replace(' ','')) for r in reader]
    else:
        # FASTA; a single sequence may span multiple lines.
        if not raw.lstrip().startswith('>'):
            raise ValueError('Input must be FASTA, CSV or TSV with sequence column; ordinary text/training file not accepted')
        seqs=[]; curr=[]
        for line in raw.splitlines():
            line=line.strip()
            if line.startswith('>'):
                if curr:seqs.append(''.join(curr).upper());curr=[]
            elif line:curr.append(line.replace(' ',''))
        if curr:seqs.append(''.join(curr).upper())
    invalid=[s for s in seqs if not 5<=len(s)<=32 or set(s)-AMINO]
    valid=[s for s in seqs if 5<=len(s)<=32 and not (set(s)-AMINO)]
    dedup=list(dict.fromkeys(valid))
    return dedup,{'raw_count':len(seqs),'valid_count':len(valid),'unique_count':len(dedup),
                  'invalid_count':len(invalid),'duplicate_count':len(valid)-len(dedup),
                  'invalid_preview':invalid[:5]}


def inspect_git_repo(repo,out):
    import subprocess
    repo=Path(repo);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    meta={'expected_origin':SOURCE_URI,'expected_tag':RELEASE_TAG,'expected_sha_prefix':RELEASE_SHA_PREFIX,
          'repo_exists':repo.exists(),'provenance_verified':False}
    if not (repo/'.git').exists():
        meta['warning']='Git working tree missing: did not assert AMPGAN v2 provenance.'
        (out/'release_provenance.json').write_text(json.dumps(meta,indent=2))
        return meta
    def git(*args):
        p=subprocess.run(['git','-C',str(repo),*args],capture_output=True,text=True)
        return p.stdout.strip() if p.returncode==0 else f'ERROR: {p.stderr[:400]}'
    meta.update({'commit':git('rev-parse','HEAD'),'origin':git('remote','get-url','origin'),
                 'tag_commit':git('rev-list','-n','1',RELEASE_TAG),
                 'head_describe':git('describe','--tags','--always'),
                 'git_status':git('status','--porcelain')})
    meta['provenance_verified']=(meta['commit'].startswith(RELEASE_SHA_PREFIX)
                                 and meta['tag_commit']==meta['commit']
                                 and 'vail-uvm/amp-gan' in meta['origin'].lower())
    filelist=[]
    for f in sorted(repo.rglob('*')):
        if not f.is_file() or '.git' in f.relative_to(repo).parts:continue
        s=f.stat().st_size
        rel=str(f.relative_to(repo))
        category='potentially_generated' if re.search(r'generated|output|results|candidate|samples',rel,re.I) else 'unknown_or_training'
        pointer=False
        if s<1000:
            try:pointer=f.read_text(errors='replace').startswith('version https://git-lfs.github.com/spec/v1')
            except OSError:pass
        filelist.append({'path':rel,'size_bytes':s,'suffix':f.suffix.lower(),'category_hint_not_proof':category,
                         'git_lfs_pointer':int(pointer),'sha256':sha256(f) if s<40*1024*1024 else 'SKIPPED_GT_40MB'})
    save_csv(out/'repository_file_inventory.csv',filelist,fieldnames=['path','size_bytes','suffix','category_hint_not_proof','git_lfs_pointer','sha256'])
    meta['file_count']=len(filelist)
    meta['potential_candidate_paths']=[x['path'] for x in filelist if x['category_hint_not_proof']=='potentially_generated'][:150]
    meta['lfs_pointer_count']=sum(x['git_lfs_pointer'] for x in filelist)
    (out/'release_provenance.json').write_text(json.dumps(meta,indent=2))
    # This does NOT certify any file as a genuine generator output.
    return meta


def ngrams(seq,n=3):return frozenset(seq[i:i+n] for i in range(max(0,len(seq)-n+1)))

def jaccard(a,b):
    aa,bb=ngrams(a),ngrams(b)
    return len(aa&bb)/len(aa|bb) if aa|bb else 0.0


def compare_sequences(ampgan,pace,hydramp,phase8,out,prov):
    import numpy as np
    from sklearn.ensemble import ExtraTreesRegressor
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    libs={'PACE_original20':pace,'HydrAMP_official_top100':hydramp,'AMPGANv2_authenticated':ampgan}
    for k,v in libs.items():
        assert len(v)==len(set(v)),f'Duplicate in {k}'
        assert all(set(s)<=AMINO and 5<=len(s)<=32 for s in v),k
    # Pre-defined, deterministic 100-sequence cap for external comparison.
    sampling={'seed':20261113,'sampling_policy':'uniform without replacement; no potency-based selection','original_count':len(ampgan)}
    if len(ampgan)>100:
        rng=np.random.default_rng(sampling['seed']);ids=sorted(rng.choice(len(ampgan),size=100,replace=False).tolist())
        libs['AMPGANv2_authenticated']=[ampgan[i] for i in ids]
        sampling['selected_original_zero_based_indices']=ids
    else:sampling['selected_original_zero_based_indices']=list(range(len(ampgan)))
    (out/'amgan_sampling_protocol.json').write_text(json.dumps(sampling,indent=2))
    z=zipfile.ZipFile(phase8)
    db=np.load(io.BytesIO(z.read('all_embeddings_and_features.npz')))
    X90=np.asarray(db['X90'],dtype=np.float32)
    seq_ref=np.asarray(db['sequences']).tolist()
    ref_lookup={s:i for i,s in enumerate(seq_ref)}
    module_path=out/'feature_engine_unmodified.py'
    module_path.write_bytes(z.read('pace_original_features.py'))
    spec=importlib.util.spec_from_file_location('pace_phase11c_features',module_path)
    f=importlib.util.module_from_spec(spec);spec.loader.exec_module(f)
    queries=[s for lib in libs.values() for s in lib]
    XX=np.asarray(f.featurize_many(queries),dtype=np.float32)
    if XX.shape!=(len(queries),90):raise RuntimeError(f'Expected 90 descriptors, got {XX.shape}')
    quant=list(csv.DictReader(io.StringIO(z.read('quantitative_predictions.csv').decode())))
    predictions={}
    for ep in ENDPOINTS+['HC50']:
        items=[r for r in quant if r['endpoint']==ep and r['representation']=='90D_PACE']
        if len(items)!=COUNTS[ep]:raise RuntimeError(f'Unexpected training rows for {ep}: {len(items)} != {COUNTS[ep]}')
        x=np.asarray([ref_lookup[r['sequence']] for r in items],dtype=int)
        y=np.asarray([float(r['observed_log10']) for r in items],dtype=float)
        model=ExtraTreesRegressor(n_estimators=300,random_state=2024,n_jobs=2,
                                  min_samples_split=2,min_samples_leaf=1,max_features=1.0)
        model.fit(X90[x],y)
        predictions[ep]=model.predict(XX)
        print('Predicted:',ep,'trained_on',len(items),'records',flush=True)
    # Indexed reference 3-mers, no O(#reference × #query) exhaustive scan.
    reference_grams=[ngrams(s) for s in seq_ref]
    inv=collections.defaultdict(list)
    for i,gr in enumerate(reference_grams):
        for token in gr:inv[token].append(i)
    def novelty(s):
        ss=ngrams(s);indices=set()
        for tri in ss:indices.update(inv.get(tri,()))
        vmax=0.0
        for j in indices:
            rr=reference_grams[j]
            intersection=len(ss&rr)
            vmax=max(vmax, intersection/(len(ss)+len(rr)-intersection))
        return vmax
    records=[];offset=0
    for library,seqs in libs.items():
        for k,s in enumerate(seqs):
            i=offset+k
            vals=np.array([predictions[ep][i] for ep in ENDPOINTS]);mean=float(vals.mean());worst=float(vals.max())
            si=float(predictions['HC50'][i]-mean)
            rec={'library':library,'candidate_id':k+1,'sequence':s,'length':len(s),
                 'net_charge':float(XX[i,1]),'hydrophobic_moment':float(XX[i,6]),
                 'known_max_3mer_jaccard':novelty(s),
                 'pred_geometric_mean_MIC_uM':float(10**mean),'pred_worst_MIC_uM':float(10**worst),
                 'pred_HC50_uM':float(10**predictions['HC50'][i]),'pred_selectivity_index':float(10**si),
                 'pass_pred_MIC32_SI10':int(10**worst<=32 and 10**si>=10)}
            for ep in ENDPOINTS:rec['pred_log10_MIC_'+ep.replace(' ','_')]=float(predictions[ep][i])
            records.append(rec)
        offset+=len(seqs)
    save_csv(out/'common_pace_model_rescoring.csv',records)
    summary=[]
    for library,seqs in libs.items():
        subset=[r for r in records if r['library']==library]
        ps=[jaccard(x,y) for x,y in combinations(seqs,2)]
        stats={'library':library,'n_scored':len(subset),'length_mean':float(np.mean([len(x) for x in seqs])),
               'length_min':min(map(len,seqs)),'length_max':max(map(len,seqs)),
               'pairwise_jaccard3_median':float(np.median(ps)) if ps else float('nan'),
               'known_max3mer_mean':float(np.mean([r['known_max_3mer_jaccard'] for r in subset])),
               'mean_pred_geometricMIC_uM':float(np.mean([r['pred_geometric_mean_MIC_uM'] for r in subset])),
               'mean_pred_worstMIC_uM':float(np.mean([r['pred_worst_MIC_uM'] for r in subset])),
               'mean_pred_selectivity':float(np.mean([r['pred_selectivity_index'] for r in subset])),
               'pass_pred_MIC32_SI10_n':sum(r['pass_pred_MIC32_SI10'] for r in subset)}
        summary.append(stats)
    save_csv(out/'common_benchmark_summary.csv',summary)
    interpretation={
      'status':'COMPUTATIONAL COMPARISON COMPLETED ONLY IF AMPGAN v2 FASTA PROVENANCE IS AUTHENTICATED',
      'input_provenance':prov,'phase8_sha256':sha256(phase8),'group_summary':summary,
      'interpretation_limitations':[
        'PACE peptides were optimized against closely related PACE reward models; PACE-model rescoring is NOT independent proof of potency or hit-rate superiority.',
        'Libraries differ in intended pathogen targets, original generator length ranges, post-selection policy and count.',
        'An analogous score from the same predictor is a model estimate, not a measured MIC or HC50.',
        'ESKAPE specificity and biological activity need paired wet-lab experiments.',
        'Because PACE original 20 has 28–30aa and HydrAMP top100 11–25aa, a three-way matched-length subset may not exist.',
        'This run is NOT a prospective equal-budget retraining or generative-model inferential benchmark.'
      ]
    }
    (out/'comparison_status.json').write_text(json.dumps(interpretation,indent=2))
    return interpretation


def archive(out,dest):
    out=Path(out);dest=Path(dest)
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED,compresslevel=8) as z:
        for f in out.rglob('*'):
            if f.is_file() and not f.name.endswith('.pyc'):z.write(f,f.relative_to(out))
    with zipfile.ZipFile(dest) as z:
        bad=z.testzip()
        if bad:raise ValueError(f'Corrupted zip member {bad}')
    print(f'OUTPUT_ZIP={dest},bytes={dest.stat().st_size}')


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--repo',default='/content/ampgan_v2_official')
    p.add_argument('--out',default='/content/PACE_Phase11C_Output')
    p.add_argument('--zip',default='/content/PACE_Phase11C_Results.zip')
    p.add_argument('--ampgan',default='')
    p.add_argument('--provenance-note',default='')
    p.add_argument('--phase8',default='')
    p.add_argument('--pace',default='')
    p.add_argument('--hydramp',default='')
    args=p.parse_args()
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    meta=inspect_git_repo(args.repo,out)
    if args.ampgan:
        if not args.provenance_note.strip():raise ValueError('Must describe the provenance of AMPGAN v2 generated peptides; no dataset auto-labeled from repo')
        pth=Path(args.ampgan)
        ampgan,checks=sequences_from_file(pth)
        info={'path':str(pth),'sha256':sha256(pth),'user_supplied_provenance':args.provenance_note,
              'status':'DECLARED, not independently authenticated by this script','quality_checks':checks,
              'expected_generator':'AMPGAN v2 release v2.0 (NOT v1 or v3)'}
        (out/'candidate_provenance_and_filters.json').write_text(json.dumps(info,indent=2))
        if len(ampgan)<20:raise ValueError('At least 20 distinct canonical AMPGAN v2 candidates needed for selected-set comparison')
        if not all([args.phase8,args.pace,args.hydramp]):
            raise ValueError('For benchmark need --phase8, --pace, --hydramp')
        pace,pc=sequences_from_file(args.pace);hydramp,hd=sequences_from_file(args.hydramp)
        if len(pace)!=20 or len(hydramp)!=100:raise ValueError(f'PACE20/HydrAMP100 expected, found {len(pace)}, {len(hydramp)}')
        compare_sequences(ampgan,pace,hydramp,args.phase8,out,info)
    else:
        (out/'comparison_status.json').write_text(json.dumps({
          'status':'SOURCE-INVENTORY ONLY; no authenticated AMPGAN v2 generated FASTA provided; NO HEAD-TO-HEAD RESULT',
          'official_tag_provenance_verified':meta['provenance_verified'],
          'required_for_scoring':['Authentic AMPGAN v2 candidate FASTA/CSV with author/source or generation-run provenance',
                                  'PACE Phase8 ESM2 results ZIP','PACE original20 FASTA','Official HydrAMP top100 FASTA'],
          'do_not_do':['Do not relabel AMPGAN training DBAsSP data as generated samples',
                       'Do not infer activity from AMPGAN paper summary statistics alone',
                       'Do not use AMPGAN v3 weights as if they were v2']},indent=2))
    (out/'runner_version.txt').write_text('PACE Phase11C analysis-only runner v1.0; no modification of source files.\n')
    archive(out,args.zip)

if __name__=='__main__':main()


In [ ]:
# Cell 3: Official source only, exact v2.0 release; skip potentially large Git LFS binaries.
import os, subprocess
repo_url='https://gitlab.com/vail-uvm/amp-gan.git'
if not (REPO/'.git').is_dir():
    env=os.environ.copy();env['GIT_LFS_SKIP_SMUDGE']='1'
    p=subprocess.run(['git','clone','--quiet','--depth','1','--branch','v2.0','--single-branch',repo_url,str(REPO)],
                     text=True,capture_output=True,env=env)
    print('Clone return code:',p.returncode)
    if p.returncode:print('Clone error (first 1200 characters):',p.stderr[:1200])
else:
    print('Repository already exists; its tag and commit will be checked explicitly.')
if (REPO/'.git').exists():
    p=subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],capture_output=True,text=True)
    print('Git commit:',p.stdout.strip())
else:
    print('Source unavailable. Audit ZIP will record this without fabricating candidate sequences.')

In [ ]:
# Cell 4: Stage 1 runs automatically, without any user upload.
import subprocess,sys
result=subprocess.run([sys.executable,'/content/pace_phase11c_runner.py',
             '--repo',str(REPO),'--out',str(OUT),
             '--zip','/content/PACE_Phase11C_Source_Audit.zip'],text=True,capture_output=True)
print(result.stdout)
if result.returncode:print('AUDIT ERROR:',result.stderr[-4000:])
assert result.returncode==0, 'Source audit failed'
import json
print(json.dumps(json.loads((OUT/'release_provenance.json').read_text()),indent=2)[:4500])
from google.colab import files
files.download('/content/PACE_Phase11C_Source_Audit.zip')

## Optional Stage 2: common sequence scoring, only after authentic AMPGAN v2 output is found

**Do not enable until you have independently checked that the candidate file contains generator outputs from the official v2.0 model (not DBAASP/AVPdb training peptides).** Record the source filename, release tag, checkpoint/generation command, and random seed when known.

If Stage 1 shows **no authenticated candidate FASTA**, leave Stage 2 disabled, upload just `PACE_Phase11C_Source_Audit.zip` to ChatGPT, and we will inspect the repository manifest before deciding how to run the actual generator.

When Stage 2 is enabled, you will need to upload two files: the **authentic AMPGAN v2 generated FASTA/CSV** and your existing **`PACE_Phase8_ESM2_results.zip`**. Embedded PACE original-20 and HydrAMP official top-100 are exact byte-for-byte copies, so you do not need to reupload those.

In [ ]:
# Exact public/user-supplied reference files packaged as byte-verified embedded data.
ASSET_DATA = {'PACE_original20.fasta': {'data_b64': 'PlBBQ0UtMDEgfCBQX0FNUD0wLjk3MCB8IG1lYW5fbG9nMTBNSUM9MC4yNzYgfCBTST04OC45IHwgbm92ZWx0eT0wLjEyMQpWUFZBUktHSVJMV1JSS0FWSFJJVlJZVFZHSUtMUAo+UEFDRS0wMiB8IFBfQU1QPTAuOTc4IHwgbWVhbl9sb2cxME1JQz0wLjM1NyB8IFNJPTczLjggfCBub3ZlbHR5PTAuMDk0ClZQVktSS0FHVkFLUElSS1dJS0xMR1JJWUZSR1ZXCj5QQUNFLTAzIHwgUF9BTVA9MC45ODEgfCBtZWFuX2xvZzEwTUlDPTAuMzY0IHwgU0k9ODQuMiB8IG5vdmVsdHk9MC4wOTEKQUxXUktHSUxHSVNWUEtLVllSUUZSUlBWUkxJS0dHCj5QQUNFLTA0IHwgUF9BTVA9MC45NzIgfCBtZWFuX2xvZzEwTUlDPTAuMzIyIHwgU0k9NjkuOSB8IG5vdmVsdHk9MC4wOTEKVlBQVktSS0FUVkFLSVJSV0lLTExHUklZRlJHVkwKPlBBQ0UtMDUgfCBQX0FNUD0wLjk2OSB8IG1lYW5fbG9nMTBNSUM9MC4zNDUgfCBTST02OC42IHwgbm92ZWx0eT0wLjEwNwpHTFdLS0dJUkxHSVNQS0tWWVJGRlJSUFZUSUtHRwo+UEFDRS0wNiB8IFBfQU1QPTAuOTc2IHwgbWVhbl9sb2cxME1JQz0wLjM3MyB8IFNJPTc1LjggfCBub3ZlbHR5PTAuMDg2CkFXS0dJVkxHSVNWUEtLVllSUUZSUlBHUkxJV1JLCj5QQUNFLTA3IHwgUF9BTVA9MC45NzIgfCBtZWFuX2xvZzEwTUlDPTAuMzQ0IHwgU0k9NjkuMyB8IG5vdmVsdHk9MC4wOTcKVlBWS1JLR1FUQUtQSVJSV0lLTExHUklZQUdWUlcKPlBBQ0UtMDggfCBQX0FNUD0wLjk4MiB8IG1lYW5fbG9nMTBNSUM9MC40MDUgfCBTST03MS4wIHwgbm92ZWx0eT0wLjExMQpJVldSS0dJR0xHSVZLS1ZZUkZGUlJQVlRJS0dHCj5QQUNFLTA5IHwgUF9BTVA9MC45NzYgfCBtZWFuX2xvZzEwTUlDPTAuMzY4IHwgU0k9NzAuNCB8IG5vdmVsdHk9MC4xMTYKVlBWR1JLQUlSUldJQVRWQUtXTklSUllJS0xMR1JJCj5QQUNFLTEwIHwgUF9BTVA9MC45NzggfCBtZWFuX2xvZzEwTUlDPTAuMzM3IHwgU0k9NTUuMyB8IG5vdmVsdHk9MC4xMDMKVlZQVkFSS0dJUkxXUlJLQVZIUkxMR1JJWUFHVlJXCj5QQUNFLTExIHwgUF9BTVA9MC45NjYgfCBtZWFuX2xvZzEwTUlDPTAuMzg3IHwgU0k9ODAuMCB8IG5vdmVsdHk9MC4wODMKQUxXUktHSUxHSVNWUEtLVllSVkFLUElSUldJS0xMCj5QQUNFLTEyIHwgUF9BTVA9MC45NjMgfCBtZWFuX2xvZzEwTUlDPTAuMzc3IHwgU0k9NzMuOSB8IG5vdmVsdHk9MC4wODMKVlBBUktHSVJMV0hSUktBVkZSSVZSUVlUVkdJS0xQCj5QQUNFLTEzIHwgUF9BTVA9MC45NzAgfCBtZWFuX2xvZzEwTUlDPTAuNDAwIHwgU0k9NjkuMSB8IG5vdmVsdHk9MC4wODkKVlBWS1JLQVFWQUtXUElSUkdJS0xMR1JJWUZSR1ZJCj5QQUNFLTE0IHwgUF9BTVA9MC45NzIgfCBtZWFuX2xvZzEwTUlDPTAuMzc2IHwgU0k9NjUuNSB8IG5vdmVsdHk9MC4wOTUKVlBWS0tLQUZWQUtQSVJSV0lLTExHUklZRlJHVkdJCj5QQUNFLTE1IHwgUF9BTVA9MC45NzQgfCBtZWFuX2xvZzEwTUlDPTAuMzY2IHwgU0k9NTEuNyB8IG5vdmVsdHk9MC4xMDMKR0xXUktHSVJRR0lTVlBLS1ZZUkZGUlJJVlRMTEdSCj5QQUNFLTE2IHwgUF9BTVA9MC45NjkgfCBtZWFuX2xvZzEwTUlDPTAuMzgzIHwgU0k9NTYuMCB8IG5vdmVsdHk9MC4xMTEKVkxXUlJHSVJMR0lTUVBLS1ZZUkZGUlJMVlRJS0dHCj5QQUNFLTE3IHwgUF9BTVA9MC45NzAgfCBtZWFuX2xvZzEwTUlDPTAuMzk4IHwgU0k9NjQuMiB8IG5vdmVsdHk9MC4xMDcKR0xXUktHSVJMSVBLS1ZZUkZGUlJQVlRJR0lLTEwKPlBBQ0UtMTggfCBQX0FNUD0wLjk1OCB8IG1lYW5fbG9nMTBNSUM9MC4zNzIgfCBTST03Ni44IHwgbm92ZWx0eT0wLjEwMApOVlBWS0FUUktQSVJSV0lLTExHUklZSVZMV1JSR0kKPlBBQ0UtMTkgfCBQX0FNUD0wLjk2MCB8IG1lYW5fbG9nMTBNSUM9MC4zNjIgfCBTST01Ny45IHwgbm92ZWx0eT0wLjExMQpHQVJLR0lSUldSUktBVkhSSVZSUVlUVkdJVkFMUAo+UEFDRS0yMCB8IFBfQU1QPTAuOTUzIHwgbWVhbl9sb2cxME1JQz0wLjM5NCB8IFNJPTcyLjMgfCBub3ZlbHR5PTAuMTExClBWS0FUUktISUZSV0lLVkxHUkxHSVZWUEtLVllSCg==', 'sha256': '6232f4cdd5c480fce9955be67f19eea36029b3e512c5743af44de9c719d76ba0'}, 'HydrAMP_official_top100.fasta': {'data_b64': 'PnNlcTEKTEtXV0tNTEtBTEtBRk1NTEFNTUdSCj5zZXEyClZXTFJMTFFXV1lRTExBTUFBTU1BCj5zZXEzCkdJS1dJTExUR1RQRk5LVFZMVFRMVEdLV1cKPnNlcTQKRkZLSUlGTFJXV1lSS1FBTEFBTUFBTU1XCj5zZXE1CkVGUlZGV1dSVllSV1JXUE5UQVdECj5zZXE2ClFXS1BXS0FNTEtRS0ZGUk1NTERBQU1BS1cKPnNlcTcKSFdLSVlSWVlGV0tXWVRFTEFNQUdUSU5JSQo+c2VxOApIS1FGUkZJUVdSWUhMTEFBS0FLV01NQVcKPnNlcTkKTUZXV0lMTktGRklWTE1NQUFES1dXCj5zZXExMApIV0ZXS01SUVdEQUtLTU1BTUxSCj5zZXExMQpQV0FXR0lZUllZUUxJS0tMTEFBS0tXQUEKPnNlcTEyCkdBV1dLV0tBV1dBS1dXS1dXQUFXTU0KPnNlcTEzCk1HV01LUUZXV0tJTkxES1dLR0lSQUtBTU0KPnNlcTE0ClZSVFBGS0lGS0FMTEdNQUhIUksKPnNlcTE1CkhWVEdSSUxLV1FZTUtMV1dLQU1NQU1BVwo+c2VxMTYKSFRXS0xRTEtLV0VNUEtFV0tXV1JNVFdSUgo+c2VxMTcKVExLTEFQS0FMUVFSV0tNRk1NTFcKPnNlcTE4ClFLTFdSUVlSWVlSRkZSWVFSRldLQURNCj5zZXExOQpWV0tMTFBWWVdQUldLSUZRUUtBRgo+c2VxMjAKSUxXS0xJTFdXS0dOTFNOSUFSUk1HCj5zZXEyMQpMUVdLRVdLUFJXV0FNTUFXS0tBV1dSUldXCj5zZXEyMgpHRktSUFlGTFJMV1dLRE1BTU1BQUtNTUtNCj5zZXEyMwpHQVNBR1dBTFdSRlJXV0lSQU1BUlNSCj5zZXEyNApNTVdWRlZSWVdLV1JXVkdWTUFSWVFZCj5zZXEyNQpZTEZMV0tMS1dXS1FSUFFLSUlMTkFLUldXCj5zZXEyNgpUUFdLTkxLV0tMV1dLR01GQVZHTUFBRERFCj5zZXEyNwpIS0FNVkFMS0tJRldSRllTRlJSWURXCj5zZXEyOApNRUVXS0lLV1dMQVNWUU5MS0tXV01NCj5zZXEyOQpNTEtLTExOV1dLRUxMU01UTExLUUFNCj5zZXEzMApXV0tMTEVETExWS1JRTUdHTU1SVkQKPnNlcTMxCkFZUllWUldMTEtXUUxLSVZSQQo+c2VxMzIKSUtMV1JUTFZMTU1MS0lNTUFXVwo+c2VxMzMKSUxXS0xMTk5MSUtXV0ZJTk5MVEVMTEVFTAo+c2VxMzQKR05XUlJQV1dSWVRJWVBSUlNBVwo+c2VxMzUKTVBZV0tOV0xEU0xXSUtBV0FHTU1XVwo+c2VxMzYKSVdGV1dLWUZXV1JGVE1BV0FER1lSCj5zZXEzNwpZS1dSV0xXV1JZRklHTU1BS04KPnNlcTM4CkZSV1FLUVdXWVdXUlJWVllEVgo+c2VxMzkKR0FXTExJQUtLSUxXUllMTktLTE1NCj5zZXE0MApHV0xRV1BMTEZJV0tMTldOTktLTEFMCj5zZXE0MQpJV05XV0tZRktBTEtXVkdES0xBVwo+c2VxNDIKV1dLV1dSTktGUU5LTFRTQUdBR0FHU1NBQQo+c2VxNDMKSFJXUllJWVJXUkZWVkcKPnNlcTQ0CkZMUExLS0FMVkZWS1JMTUFGTEFLUlkKPnNlcTQ1CkhBUktMV1dOUllGRllSUldZCj5zZXE0NgpJS0xMSUxLS0xBTVdNTVdBQVdXUlcKPnNlcTQ3CllXV1JXUFBXV0tZUVFSTkFNCj5zZXE0OApHS1dSV05XS0tBQU1NUlkKPnNlcTQ5CkdSRlJZRlJXV0ZUV0lWSFJEUgo+c2VxNTAKSEtBTFdXTExLUkZUQUZHTUFHQU1NQU1MSwo+c2VxNTEKRldMS1dXS0xMSVRMTFJXS0RBTVcKPnNlcTUyClBXUVlGV0tUVlZHVlZIUFlIUFFQUFFLS1cKPnNlcTUzCk1GV1dLV1dLV1dMQUVJRlFLQURNCj5zZXE1NApJV1dGV1dSUVlSRklXTEFNV1cKPnNlcTU1CkdXUlJJVldXUkxFTFdSS0FMCj5zZXE1NgpMTldLTExTUFdXTFFSWUhJSVlLV1IKPnNlcTU3CkFMS0lUU0dGV1dLRklWR01BQQo+c2VxNTgKR1JXSUtMUldXUVlJV05LS0lQUUVBCj5zZXE1OQpWS0xLTFdMS1dXS1dBTU1JTU1LQURXCj5zZXE2MApGUEtXUllJTFZMSElLUlZBQQo+c2VxNjEKR01XTEtRTElJUkVMV1dLS0xMQUxMS0tXTAo+c2VxNjIKU0lSV1dSWVJXV1lURExHQURFTFJXTQo+c2VxNjMKQUFXUk5MS0xXV1FMV0RNQVJFTFJXVwo+c2VxNjQKU1JMTlNGRlJSV0ZGVlBSWVZWTUgKPnNlcTY1CldXTUxMV0tRWVJZUUtJRlZMS0xBTVIKPnNlcTY2ClFXUldXUllLV1BITEtRUVRTVFNTR1JBS1cKPnNlcTY3CkZQV1RJTklSV1dSV1dMTVZJTVcKPnNlcTY4ClNXS0tMQVdXS1dBQUxBTU1BV1dEVwo+c2VxNjkKR1JTUklQUkdSWVFNV0tLQU0KPnNlcTcwCk1QV05LV0tMTFdXCj5zZXE3MQpWVktXUFdRV0tGS1dMTEFGRkxMTkxBTVJXCj5zZXE3MgpQUUhJUktZUVdMSAo+c2VxNzMKSFdMV0xMS1dQUlFZRFdXUlJXRVdXCj5zZXE3NApGSUtSWU5QWVBSSFZJSUZSUlBOV1cKPnNlcTc1CllWVkxSUUlXV1BRRlJGWURLTFIKPnNlcTc2ClNGV1JSV05WVkhIVkhOU1JJQUFSTAo+c2VxNzcKU0xXS0xLQUFMS0lLTEFNQUtLTUEKPnNlcTc4Ck1XV1JXS1dJTE5XSVZTREFJUldXCj5zZXE3OQpUSVdRV1JLTU1FTFdLS0xXV0FBV0FSCj5zZXE4MApNS0lMTlJJVElMTEtXV0xBU1lFTAo+c2VxODEKS1dXUURNV0FJQVdLS0FMTEFLS0RMCj5zZXE4MgpTTktXV1JRWVJZTExXTEFBTQo+c2VxODMKUExSV0xLV05QR1BBQUdWR01IUldXCj5zZXE4NApTSUxXS0xOS1dTSFdXS1FBCj5zZXE4NQpGSVdJWVJXV0tJTFdSWVNZCj5zZXE4NgpIWUVOTFdMTFdXS1dXQVZTQUtSVwo+c2VxODcKUFJJV1JRSUZWRktXCj5zZXE4OApHS0xLTlZRTEtMTEtXS1NWRkhBTFcKPnNlcTg5CkZSTEtOUVdQS0xSV1dLV0tBTUFBCj5zZXE5MApJR1JXRlJZUllWVklGTExJR0FBV0tNV0tXCj5zZXE5MQpMUldSUUlWSEtXUVdXRlBLTFRTTEFOCj5zZXE5MgpMSEdMU1RZUldXUldXS0dSCj5zZXE5MwpWV1JXV0ZIS0tJVExLS1dBTUFBRERLV1dLCj5zZXE5NApBUExSUkxZSVdMREtXV0tXV0FLCj5zZXE5NQpQUk1BVlFLTFZTUldSTU1JQUFTS0xLRE0KPnNlcTk2CkdGTFJRV1dSUllHTVZSVwo+c2VxOTcKTVBSUkZXV1JXUllQTEtMUkFXCj5zZXE5OApGV1JZSUZZSEdGVlZMVk1NQUFNTUFHQU1BCj5zZXE5OQpHVExLS0xMV1dLQUxSWUxHV0sKPnNlcTEwMApBV0tXUldXUklITFdSCg==', 'sha256': '146663dfdaa4635b779cfc83d942420520d51bc0877bea6271b7100cbe638dc5'}}
print("Exact original PACE20 and official HydrAMP100 reference FASTAs bundled.")

In [ ]:
# Cell 6: ENABLE ONLY WITH AUTHENTIC AMPGAN v2 GENERATED OUTPUT.
RUN_OPTIONAL_BENCHMARK = False
GENERATED_FILE = '' # e.g. 'ampgan_v2_generated_candidates.fasta'
GENERATION_PROVENANCE = '' # e.g. 'official v2.0 tag, checkpoint [id], generation script [name], seed [n]'

if RUN_OPTIONAL_BENCHMARK:
    assert GENERATED_FILE and GENERATION_PROVENANCE.strip(), 'Source path AND provenance note are required.'
    import base64,hashlib
    from google.colab import files
    # Upload generated source and Phase8 ZIP if not already on runtime.
    required=[Path('/content')/GENERATED_FILE,Path('/content/PACE_Phase8_ESM2_results.zip')]
    if any(not q.exists() for q in required):
        print('Upload AMPGAN v2 output file and original PACE_Phase8_ESM2_results.zip:')
        uploaded=files.upload()
    assert all(q.exists() for q in required),f'Missing files: {[str(q) for q in required if not q.exists()]}'
    INPUT=Path('/content/pace_phase11c_reference_assets');INPUT.mkdir(exist_ok=True)
    asset_data=ASSET_DATA
    for name,info in asset_data.items():
        b=base64.b64decode(info['data_b64'])
        assert hashlib.sha256(b).hexdigest()==info['sha256'],name
        (INPUT/name).write_bytes(b)
    p=subprocess.run([sys.executable,'/content/pace_phase11c_runner.py',
        '--repo',str(REPO),'--out',str(OUT), '--zip','/content/PACE_Phase11C_Results.zip',
        '--ampgan',str(required[0]),'--provenance-note',GENERATION_PROVENANCE,
        '--phase8',str(required[1]),'--pace',str(INPUT/'PACE_original20.fasta'),
        '--hydramp',str(INPUT/'HydrAMP_official_top100.fasta')],text=True,capture_output=True)
    print(p.stdout)
    if p.returncode:print('SCORING ERROR:',p.stderr[-6000:])
    assert p.returncode==0,'Optional scoring incomplete; share traceback instead of claiming results'
    files.download('/content/PACE_Phase11C_Results.zip')
else:
    print('Optional head-to-head scoring deliberately disabled pending authentic v2 generated sequences.')

### Upload the result

For your first run, upload **`PACE_Phase11C_Source_Audit.zip`**. This ZIP includes the official release verification, a file inventory and an explicit statement of whether AMPGAN v2 sequences were authenticated. It does **not** claim AMPGAN generation completed.

If you independently obtain authenticated v2 sequences and run Stage 2, upload **`PACE_Phase11C_Results.zip`** as well. All outcomes will be labeled *model predictions*, not MIC/HC50 measurements.